In [1]:
from utils_00 import *
# exec(open('c00_utils.py').read())

pn.extension()  # 启用 Jupyter 支持
pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', 100)

In [2]:
native_exchange_data_dir = os.path.join(base_dir, "Native_Exchange_Data")
temporary_experimental_data_dir = os.path.join(base_dir, "Temporary_Experimental_Data")
pro = ts.pro_api(secret_dict['tushare']['api_key'])

def get_exchange_variety_map():
    # 交易所日历路径 NED_dimension_exchange_calendar_path
    NED_dimension_exchange_calendar_path = os.path.join(native_exchange_data_dir, 'dimension_exchange_calendar.csv')
    dimension_exchange_calendar = pd.read_csv(NED_dimension_exchange_calendar_path, dtype = {'date': str})

    # 获取 交易所-品种 表单
    exchange_list = [col for col in dimension_exchange_calendar.columns if col != 'date'] # 交易所列表
    map_list = []
    for exchange in exchange_list:

        exchange_variety_map = pd.DataFrame()
        for attempt in range(3):
            try:
                exchange_variety_map = pro.fut_basic(exchange = exchange, fut_type = "1", fields = ["fut_code"])
                break
            except Exception as e:
                print(f"第 {attempt + 1} 次获取 {exchange} 品种信息失败: {e}")

        exchange_variety_map = exchange_variety_map.drop_duplicates()
        exchange_variety_map["exchange"] = exchange
        map_list.append(exchange_variety_map)

    exchange_variety_map = pd.concat(map_list, ignore_index = True)
    exchange_variety_map = exchange_variety_map.groupby('exchange')['fut_code'].apply(list)
    exchange_variety_map = exchange_variety_map.to_dict()
    return exchange_variety_map

if __name__ == '__main__':
    exchange_variety_map = get_exchange_variety_map()
    exchange_variety_map.pop("CZCE")
    exchange_variety_map.pop("DCE")
    print(exchange_variety_map)

{'CFFEX': ['T', 'TF', 'IF', 'IM', 'IC', 'IH', 'TS', 'TL'], 'GFEX': ['LC', 'SI', 'PS', 'PD', 'PT'], 'INE': ['NR', 'SC', 'LU', 'BC', 'SCTAS', 'EC'], 'SHFE': ['AO', 'NI', 'AL', 'HC', 'BU', 'RU', 'FU', 'CU', 'AG', 'ZN', 'AU', 'PB', 'SS', 'RB', 'SN', 'WR', 'SP', 'BR', 'AD', 'OP']}


In [3]:
exchange = None
variety = None

if __name__ == '__main__':
    # 交易所选择组件
    exchange_radio = widgets.RadioButtons(
        options = list(exchange_variety_map.keys()),
        description = '交易所选择 exchange:',
        layout = {'width': '400px'}
    )

    # 品种选择组件（初始禁用）
    species_toggle = widgets.ToggleButtons(
        options = [],
        description = '品种选择 variety:',
        disabled = True,
        button_style = '',
        style = {'button_width': 'auto'},
        layout = widgets.Layout(width = '600px'),
    )

    # 输出提示区域
    out = widgets.Textarea(
        value = '请先选择交易所，然后选择品种',
        layout = {'width': '400px', 'height': '50px'},
        disabled = True
    )

    display(widgets.VBox([exchange_radio, species_toggle, out]))

    def on_exchange_change(change):
        """当交易所改变时，更新可选品种"""
        global exchange
        if change['name'] == 'value':
            exchange = change['new']
            varieties = exchange_variety_map.get(exchange, [])
            species_toggle.options = sorted(varieties)
            species_toggle.disabled = len(varieties) == 0
            if varieties:
                species_toggle.value = varieties[0]  # 默认选第一个
                out.value = f'已选择交易所：{exchange}，请选择品种'
            else:
                out.value = f'交易所 {exchange} 无可用品种'

    def on_species_change(change):
        """当品种改变时，记录到全局变量"""
        global variety
        if change['name'] == 'value' and species_toggle.options:
            variety = change['new']
            out.value = f'已选择交易所 exchange: {exchange}\n已选择品种 variety: {variety}'

    # 绑定事件 ["SCTAS", "SI", "FU", "AU", "SS", "WR"]
    exchange_radio.observe(on_exchange_change, names = 'value')
    species_toggle.observe(on_species_change, names = 'value')

In [4]:
dataset_range_boundary = ""  # 选择的时间范围边界字符串

if __name__ == '__main__':

    # 临时实验数据品种文件夹路径 TED_variety_path
    TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))
    dataset_range = os.listdir(TED_variety_path)
    dataset_range = sorted(dataset_range)

    dataset_range_buttons = []  # 创建数据集范围边界按钮容器
    for range_file in dataset_range:  # 为每个范围文件创建按钮
        btn = widgets.Button(
            description = range_file,
            layout = {'width': '530px', 'height': '25px'},  # 宽度较大以显示完整文件名
            button_style = ''
        )
        dataset_range_buttons.append(btn)

    dataset_range_button_column = widgets.VBox(dataset_range_buttons) # Horizontal Box & Vertical Box

    # 输出显示区域
    range_display_output = widgets.Textarea(
        value = 'dataset_range_boundary: 未选择',
        layout = {'width': '530px', 'height': '30px'},
        disabled = True
    )

    # 显示所有组件
    display(widgets.VBox([
        widgets.Label('数据集范围边界 dataset_range_boundary:'),
        dataset_range_button_column,
        range_display_output
    ]))

    def update_range_display():
        """更新显示区域"""
        global dataset_range_boundary
        range_display_output.value = f'dataset_range_boundary: {dataset_range_boundary}'

    def set_dataset_range_boundary(range_file):
        """设置数据集范围边界"""
        global dataset_range_boundary
        dataset_range_boundary = range_file
        update_range_display()

    # 事件处理函数
    def on_dataset_range_button_click(btn):
        """数据集范围按钮 点击事件"""
        set_dataset_range_boundary(btn.description)

    # 绑定事件
    for btn in dataset_range_buttons:
        btn.on_click(on_dataset_range_button_click)

    # 初始化
    update_range_display()

In [5]:
tick_granularity = 0  # 采样颗粒度 全局变量
resample_axis = None  # 采样轴 全局变量

if __name__ == '__main__':

    # 创建 tick_granularity 输入框组件
    tick_granularity_input = widgets.IntText(
        value = 0,
        description = 'tick_granularity:',
        layout = {'width': '250px', 'height': '25px'},
        style = {'description_width': '90px'},
        continuous_update = False
    )

    # 创建 tick_granularity 按钮容器
    granularity_buttons = []
    for i in [i for i in range(0, 181, 15)]:
        btn = widgets.Button(
            description = str(i),
            layout = {'width': '45px', 'height': '25px'}
        )
        btn.value = i
        granularity_buttons.append(btn)
    granularity_button_row = widgets.HBox(granularity_buttons)

    # tick_granularity 输出显示区域
    granularity_out = widgets.Textarea(
        value = '当前 tick_granularity: 0',
        layout = {'width': '400px', 'height': '30px'},
        disabled = True
    )

    # 创建 resample_axis 按钮
    axis_buttons = []
    axis_options = ['time_axis_data', 'volume_axis_data', 'money_axis_data']  # 重采样刻度选项
    for option in axis_options:
        btn = widgets.Button(
            description = option,
            layout = {'width': '135px', 'height': '25px'}
        )
        btn.value = option
        axis_buttons.append(btn)
    axis_button_row = widgets.HBox(axis_buttons)

    # resample_axis 显示区域
    axis_out = widgets.Textarea(
        value = '当前 resample_axis: None',
        layout = {'width': '400px', 'height': '30px'},
        disabled = True
    )

    # 显示所有组件
    display(widgets.VBox([tick_granularity_input, granularity_button_row, granularity_out, axis_button_row, axis_out]))

    def update_display(value):
        """更新显示"""
        global tick_granularity
        tick_granularity = value
        granularity_out.value = f'当前 tick_granularity: {tick_granularity}'
        tick_granularity_input.value = value

    def update_axis_display(value):
        """更新 resample_axis 显示"""
        global resample_axis
        resample_axis = value
        axis_out.value = f'当前 resample_axis: {resample_axis}'

    def on_input_change(change):
        """当输入框值改变时"""
        if change['name'] == 'value':
            new_value = change['new']
            if 0 <= new_value <= 600:
                update_display(new_value)
            else:
                tick_granularity_input.value = tick_granularity  # 恢复原值

    def on_button_click(btn):
        """当按钮点击时"""
        new_value = btn.value
        update_display(new_value)

    def on_axis_button_click(btn):
        """当 resample_axis 按钮点击时"""
        new_value = btn.value
        update_axis_display(new_value)

    # 绑定输入框事件
    tick_granularity_input.observe(on_input_change, names ='value')

    # 绑定所有按钮事件
    for btn in granularity_buttons:
        btn.on_click(on_button_click)

    # 绑定 resample_axis 按钮事件
    for btn in axis_buttons:
        btn.on_click(on_axis_button_click)

In [6]:
lookback_months = 0  # 回溯月份数 全局变量
split_index = None  # 切割点 index 全局变量
split_time_str = None  # 切割时间字符串 全局变量

if __name__ == '__main__':

    TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))  # 临时实验数据品种文件夹路径 TED_variety_path
    dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary)  # 时间范围文件夹路径 dataset_range_boundary_path
    tick_granularity_resample_path = os.path.join(dataset_range_boundary_path, f'tick_granularity_{tick_granularity}')  # 采样刻度文件路径 tick_granularity_resample_path
    resample_axis_data_path = os.path.join(tick_granularity_resample_path, resample_axis)  # 驱动轴数据文件路径 resample_axis_data_path

    tick_type_dict = {
        'time_axis_data': 'Nrow',
        'volume_axis_data': 'volume',
        'money_axis_data': 'money'
    }
    if resample_axis in tick_type_dict.keys():
        tick_type = tick_type_dict[resample_axis]
    else: raise ValueError

    # 主力片段拼接重采样时间索引路径 main_continuous_resampled_date_index_path
    main_continuous_resampled_date_index_path = os.path.join(resample_axis_data_path, rf"main_continuous_{tick_type}_date_index.csv")
    main_continuous_resampled_date_index = pd.read_csv(main_continuous_resampled_date_index_path, dtype = {'date': str})

    # 创建 lookback_months 输入框组件
    lookback_months_input = widgets.IntText(
        value = 0,
        description = 'lookback_months:',
        layout = {'width': '250px', 'height': '25px'},
        style = {'description_width': '100px'},
        continuous_update = False
    )

    # 创建 lookback_months 按钮容器（1~24）
    months_buttons = []
    for i in range(1, 25):
        btn = widgets.Button(
            description = str(i),
            layout = {'width': '45px', 'height': '25px'}
        )
        btn.value = i
        months_buttons.append(btn)
    first_row = widgets.HBox(months_buttons[:12])   # 前12个按钮
    second_row = widgets.HBox(months_buttons[12:])  # 后12个按钮
    months_button_row = widgets.VBox([first_row, second_row])

    # 输出显示区域：包含回溯月份、切割index、切割时间
    split_info_out = widgets.Textarea(
        value = '当前 lookback_months: \nsplit_index: None \nsplit_time_str: None',
        layout = {'width': '600px', 'height': '60px'},
        disabled = True
    )

    # 显示所有组件
    display(widgets.VBox([lookback_months_input, months_button_row, split_info_out]))

    def update_split_info():
        """根据 lookback_months 和 main_continuous_resampled_date_index 计算并更新 split_index 与 split_time_str"""
        global lookback_months, split_index, split_time_str

        if lookback_months <= 0 or main_continuous_resampled_date_index.empty:
            split_index = None
            split_time_str = None
        else:
            # 获取最后一行的时间字符串并转换为 datetime
            last_time_str = main_continuous_resampled_date_index['date'].iloc[-1]
            last_dt = pd.to_datetime(last_time_str)

            # 回溯指定月份数（注意：pandas 的 DateOffset 支持月份偏移）
            target_dt = last_dt - pd.DateOffset(months=lookback_months)

            # 找到最接近但不大于 target_dt 的行的 index
            date_series = pd.to_datetime(main_continuous_resampled_date_index['date'])
            valid_mask = date_series <= target_dt
            if valid_mask.any():
                split_index = valid_mask[::-1].idxmax()  # 最后一个满足条件的 index
                split_time_str = main_continuous_resampled_date_index['date'].iloc[split_index]
            else:
                split_index = 0
                split_time_str = main_continuous_resampled_date_index['date'].iloc[0]

        # 更新显示
        split_info_out.value = f'当前 lookback_months: {lookback_months} \nsplit_index: {split_index} \nsplit_time_str: {split_time_str}'

    def on_months_input_change(change):
        """当输入框值改变时"""
        if change['name'] == 'value':
            new_value = change['new']
            if 0 <= new_value <= 24:
                global lookback_months
                lookback_months = new_value
                lookback_months_input.value = new_value
                update_split_info()
            else:
                lookback_months_input.value = lookback_months  # 恢复原值

    def on_months_button_click(btn):
        """当按钮点击时"""
        global lookback_months
        lookback_months = btn.value
        lookback_months_input.value = lookback_months
        update_split_info()

    # 绑定输入框事件
    lookback_months_input.observe(on_months_input_change, names = 'value')

    # 绑定所有按钮事件
    for btn in months_buttons:
        btn.on_click(on_months_button_click)

    # 初始化显示
    update_split_info()

In [7]:
# 临时实验数据品种文件夹路径 TED_variety_path
TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))

# 时间范围文件夹路径 dataset_range_boundary_path
dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary)

# 采样刻度文件路径 tick_granularity_resample_path
tick_granularity_resample_path = os.path.join(dataset_range_boundary_path, f'tick_granularity_{tick_granularity}')

# 驱动轴数据文件路径 resample_axis_data_path
resample_axis_data_path = os.path.join(tick_granularity_resample_path, resample_axis)



tick_type_dict = {
    'time_axis_data': 'Nrow',
    'volume_axis_data': 'volume',
    'money_axis_data': 'money'
}
if resample_axis in tick_type_dict.keys():
    tick_type = tick_type_dict[resample_axis]
else: raise ValueError

# 主力片段拼接重采样时间索引路径 main_continuous_resampled_date_index_path
main_continuous_resampled_date_index_path = os.path.join(resample_axis_data_path, f'main_continuous_{tick_type}_timeseries.csv')
main_continuous_resampled_date_index = pd.read_csv(main_continuous_resampled_date_index_path, dtype = {'date': str})

In [8]:
# 时间序列特征文件夹路径 TS_dynamic_metrics_path
TS_dynamic_metrics_path = os.path.join(resample_axis_data_path, f'TS_dynamic_metrics')

# 近月价格多个跨步差分路径 near_month_price_diffed_backward_path
near_month_price_diffed_backward_path = os.path.join(TS_dynamic_metrics_path, 'near_month_price_diffed_backward.csv')
near_month_price_diffed_backward = pd.read_csv(near_month_price_diffed_backward_path)
# near_month_price_diffed_backward.info()
# print('print(near_month_price_diffed_backward.head())')
# print(near_month_price_diffed_backward.head())

# 远月价格多个跨步差分路径 far_month_price_diffed_backward_path
far_month_price_diffed_backward_path = os.path.join(TS_dynamic_metrics_path, 'far_month_price_diffed_backward.csv')
far_month_price_diffed_backward = pd.read_csv(far_month_price_diffed_backward_path)
# far_month_price_diffed_backward.info()
# print('print(far_month_price_diffed_backward.head())')
# print(far_month_price_diffed_backward.head())

# 近主力远月价格多个跨步差分路径 near_main_near_spread_diffed_backward_path
near_main_near_spread_diffed_backward_path = os.path.join(TS_dynamic_metrics_path, 'near_main_near_spread_diffed_backward.csv')
near_main_near_spread_diffed_backward = pd.read_csv(near_main_near_spread_diffed_backward_path)
# near_main_near_spread_diffed_backward.info()
# print('print(near_main_near_spread_diffed_backward.head())')
# print(near_main_near_spread_diffed_backward.head())

# 近主力远月价格多个跨步差分路径 near_main_far_spread_diffed_backward_path
near_main_far_spread_diffed_backward_path = os.path.join(TS_dynamic_metrics_path, 'near_main_far_spread_diffed_backward.csv')
near_main_far_spread_diffed_backward = pd.read_csv(near_main_far_spread_diffed_backward_path)
# near_main_far_spread_diffed_backward.info()
# print('print(near_main_far_spread_diffed_backward.head())')
# print(near_main_far_spread_diffed_backward.head())

In [15]:
# 时间序列特征文件夹路径 TS_dynamic_metrics_path
TS_dynamic_metrics_path = os.path.join(resample_axis_data_path, f'TS_dynamic_metrics')

# 主力连续重采样收盘价多个跨步差分路径 main_continuous_resampled_timeseries_close_diffed_backward_path
main_continuous_resampled_timeseries_close_diffed_backward_path = os.path.join(TS_dynamic_metrics_path, 'main_continuous_resampled_timeseries_close_diffed_backward.csv')
main_continuous_resampled_timeseries_close_diffed_backward = pd.read_csv(main_continuous_resampled_timeseries_close_diffed_backward_path)
# main_continuous_resampled_timeseries_close_diffed_backward.info()
# print('print(main_continuous_resampled_timeseries_close_diffed_backward.head())')
# print(main_continuous_resampled_timeseries_close_diffed_backward.head())

# 主力连续重采样收盘价一阶差分多个窗口长度方差路径 main_continuous_resampled_timeseries_close_variance_path
main_continuous_resampled_timeseries_close_variance_path = os.path.join(TS_dynamic_metrics_path, 'main_continuous_resampled_timeseries_close_variance.csv')
main_continuous_resampled_timeseries_close_variance = pd.read_csv(main_continuous_resampled_timeseries_close_variance_path)
# main_continuous_resampled_timeseries_close_variance.info()
# print('print(main_continuous_resampled_timeseries_close_variance.head())')
# print(main_continuous_resampled_timeseries_close_variance.head())

# 主力连续重采样收盘价一阶差分多个窗口长度样本熵路径 main_continuous_resampled_timeseries_close_sampen_path
main_continuous_resampled_timeseries_close_sampen_path = os.path.join(TS_dynamic_metrics_path, 'main_continuous_resampled_timeseries_close_sampen.csv')
main_continuous_resampled_timeseries_close_sampen = pd.read_csv(main_continuous_resampled_timeseries_close_sampen_path)
# main_continuous_resampled_timeseries_close_sampen.info()
# print('print(main_continuous_resampled_timeseries_close_sampen.head())')
# print(main_continuous_resampled_timeseries_close_sampen.head())

# 主力连续重采样收盘价多个窗口长度z值路径 main_continuous_resampled_timeseries_close_zscore_path
main_continuous_resampled_timeseries_close_zscore_path = os.path.join(TS_dynamic_metrics_path, 'main_continuous_resampled_timeseries_close_zscore.csv')
main_continuous_resampled_timeseries_close_zscore = pd.read_csv(main_continuous_resampled_timeseries_close_zscore_path)
# main_continuous_resampled_timeseries_close_zscore.info()
# print('print(main_continuous_resampled_timeseries_close_zscore.head())')
# print(main_continuous_resampled_timeseries_close_zscore.head())

In [17]:
# 同步压缩变换 SWT 数据路径 CWT_SWT_path
CWT_SWT_path = os.path.join(resample_axis_data_path, f'CWT_SWT')

# SWT 实部数据路径 SWT_real_path
SWT_real_path = os.path.join(CWT_SWT_path, f'SWT_real.csv')
SWT_real = pd.read_csv(SWT_real_path)

# SWT 虚部数据路径 SWT_imag_path
SWT_imag_path = os.path.join(CWT_SWT_path, f'SWT_imag.csv')
SWT_imag = pd.read_csv(SWT_imag_path)

# SWT 幅值数据路径 SWT_magnitude_path
SWT_magnitude_path = os.path.join(CWT_SWT_path, f'SWT_magnitude.csv')
SWT_magnitude = pd.read_csv(SWT_magnitude_path)

# SWT 相位数据路径 SWT_phase_path
SWT_phase_path = os.path.join(CWT_SWT_path, f'SWT_phase.csv')
SWT_phase = pd.read_csv(SWT_phase_path)

# SWT 频率映射表路径 frequencies_map_path
frequencies_map_path = os.path.join(CWT_SWT_path, f'frequencies_map.csv')
frequencies_map = pd.read_csv(frequencies_map_path)

In [8]:
import torch
from torch.utils.data import DataLoader, TensorDataset

import pytorch_lightning as pl
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold

import gc  # 垃圾回收模块 Garbage Collector
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子以确保可重复性
random_state = 42
torch.manual_seed(random_state) # PyTorch 随机种子 manual_seed()
np.random.seed(random_state)  # Numpy 随机种子 random.seed()

from utils_01_VAE import VAE, train_vae_with_cv, train_final_VAE_model, save_lightning_vae, load_lightning_vae

W0403 06:26:39.645000 30208 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


2.11.0+cu130
使用设备: cuda


In [17]:
def prepare_vae_data_and_paths(data_set, data_set_name, VAE_dimension_reduction_path):
    """ 准备 VAE 降维所需的数据和路径配置 """

    train_data = data_set.iloc[ :split_index].dropna().values
    test_data = data_set.iloc[split_index: ].values

    # 驱动轴数据文件路径 resample_axis_data_path
    resample_axis_data_path = os.path.join(tick_granularity_resample_path, resample_axis)

    # 特定数据降维路径 Specific_data_VAE_dimension_reduction_path
    Specific_data_VAE_dimension_reduction_path = os.path.join(VAE_dimension_reduction_path, f'{data_set_name}_VAE_dimension_reduction')
    os.makedirs(Specific_data_VAE_dimension_reduction_path, exist_ok = True)

    # 变分自编码器降维函数 模型参数与权重保存路径 VAE_dimension_reduction_params_path
    VAE_dimension_reduction_params_path = os.path.join(Specific_data_VAE_dimension_reduction_path, f'VAE_dimension_reduction_params.pth')

    # 变分自编码器降维函数 模型类路径 VAE_dimension_reduction_class_path
    VAE_dimension_reduction_class_path = os.path.join(Specific_data_VAE_dimension_reduction_path, f'VAE_dimension_reduction_class.pkl')

    # 潜在空间数据路径 Specific_data_latent_features_path
    Specific_data_latent_features_path = os.path.join(Specific_data_VAE_dimension_reduction_path, f'{data_set_name}_latent_features.csv')

    return (
        train_data,
        test_data,
        Specific_data_VAE_dimension_reduction_path,
        VAE_dimension_reduction_params_path,
        VAE_dimension_reduction_class_path,
        Specific_data_latent_features_path
    )

In [18]:
def configure_vae_hyperparameters(train_data):
    """ 配置 VAE 超参数搜索空间 """

    def generate_hidden_configs(n_feat, max_layers = 3):
        # 隐藏层维度设计，漏斗形递减
        # 首层 ≥ 输入维，末层 ≥ 潜在维上限的 2 倍，中间逐层减半

        configs = []
        first_layers = [min(512, max(32, n_feat * i)) for i in [1, 2, 3]] # 候选首层维度：输入维度的 1x, 2x, 3x，但不超过 512

        for first in first_layers:
            configs.append([first, max(16, first // 2)]) # 2 层结构：[first, max(16, first//2)]
            if max_layers >= 3:
                second = max(32, first // 2)
                third = max(16, second // 2)
                configs.append([first, second, third])  # 3 层结构：逐层减半

        unique_configs = []
        [unique_configs.append(c) for c in configs if c not in unique_configs] # 去重并限制规模
        return unique_configs[:4]  # 保留 4 种结构

    n_features = train_data.shape[1]
    hidden_dims_pool = generate_hidden_configs(n_features)  # 结构参数

    latent_min = max(4, int(n_features * 0.15)) # 下限：max(4, n_features // 6)  约 1/6，但至少 4 维
    latent_max = max(16, int(n_features * 0.5)) # 上限：max(16, n_features // 2)  约 1/2，信息保留足够  # 弃用潜在空间寻优，统一使用 1/6

    # 定义网格搜索超参数寻优的 最大参数组合数 max_combinations 与 用于穷举组合的参数字典 param_grid 超参数搜索空间
    param_grid = {
        'hidden_dims': hidden_dims_pool,  # 结构参数
        'latent_dim': (latent_min, latent_min, 'int'),  # 潜在空间：输入维度的 15% ~ 50%（即约 1/6 到 1/2）  # 弃用潜在空间维度数寻优，统一使用 1/6
        'beta': (0.000001, 0.000001, 'log'),  # β-VAE：控制解耦程度，小数据取小值避免过度正则
        'dropout_rate': (0.05, 0.3, 'float'),  # Dropout：维度较低时适当降低避免信息瓶颈
        'lr': (1e-4, 1e-3, 'log'),
        'weight_decay': (1e-6, 1e-4, 'log'),
        'batch_size': (128, 128, 'int'),
    }

    # 设置 PyTorch Lightning 回调函数 Callbacks
    from pytorch_lightning.callbacks.early_stopping import EarlyStopping
    callbacks = [
        EarlyStopping(  # 早停，防止模型过拟合，当验证集性能不再提升时自动停止训练
            monitor = 'val_loss',  # 指定监控对象，val_loss: 验证集损失
            patience = 12,          # 容忍 epoch 数，验证损失 patience 个 epoch 不改善则停止
            mode = 'min',          # 优化方向：minimize 越小越好
            verbose = False        # 是否打印日志以显式早停信息
        ),
        # ModelCheckpoint(    # 模型检查点，在训练过程中保存最佳模型权重
        #     monitor = 'val_loss',  # 指定监控对象，val_loss: 验证集损失
        #     dirpath = f'./checkpoints/params_{param_id}_fold_{fold_idx}',
        #     filename = 'best_model',
        #     save_top_k = 1,
        #     mode = 'min'
        # )
    ]

    return (
        param_grid,
        callbacks
    )

In [19]:
def run_vae_cross_validation(train_data, n_splits, param_grid, callbacks, max_combinations):
    """
    执行 VAE 交叉验证寻优
    """
    all_results, best_params, best_score = train_vae_with_cv(
        train_data,
        n_splits,
        param_grid, # 超参数空间选项
        callbacks,
        max_combinations,
        random_state = 42
    )

    return (
        all_results,
        best_params,
        best_score
    )

In [20]:
def train_and_persist_vae_model(
    train_data, test_data, best_params,
    VAE_dimension_reduction_params_path,
    VAE_dimension_reduction_class_path
):
    """ 训练最终 VAE 模型并持久化保存 """

    vae_model, scaler = train_final_VAE_model(
        train_data,
        test_data,
        best_params
    )
    #%%
    # 变分自编码器降维函数 VAE_dimension_reduction_model
    VAE_dimension_reduction_model = vae_model
    print(f"变分自编码器降维模型 VAE_dimension_reduction_model: {VAE_dimension_reduction_model}")

    # 数据缩放器
    print('数据缩放器:', scaler)

    save_lightning_vae(
        VAE_dimension_reduction_model, VAE, scaler,
        VAE_dimension_reduction_params_path, VAE_dimension_reduction_class_path
    )  # 保存模型

    VAE_dimension_reduction_model, scaler = load_lightning_vae(
        VAE_dimension_reduction_params_path, VAE_dimension_reduction_class_path, device = 'cpu'
    )  # 加载模型

    print('数据缩放器:', scaler)
    print(f"加载的模型: {VAE_dimension_reduction_model}")

    return (
        VAE_dimension_reduction_model,
        scaler
    )

In [21]:
def extract_and_save_latent_features(
    vae_model,
    scaler,
    data_set,
    data_set_name,
    Specific_data_latent_features_path,
    main_continuous_resampled_date_index
):
    """ 提取潜在特征并保存到文件 """

    # 提取潜在特征
    def reduce_dimension(vae_model, data_tensor):
        """使用 VAE 提取潜在特征"""
        vae_model.eval()
        with torch.no_grad():  # 分批处理，避免内存不足
            batch_size = 1024
            latent_list = []

            for i in range(0, len(data_tensor), batch_size):
                batch = data_tensor[i:i + batch_size]
                latent = vae_model.encode_latent(batch.to(vae_model.device))
                latent_list.append(latent.cpu())

            return torch.cat(latent_list, dim = 0).numpy()

    # 对整个数据集降维
    print("进行降维...")
    X_all_scaled = scaler.transform(data_set.values.astype(np.float64))
    X_all_tensor = torch.FloatTensor(X_all_scaled)

    latent_features = reduce_dimension(vae_model, X_all_tensor)
    latent_features = pd.DataFrame(
        latent_features,
        columns = [f'{data_set_name}_latent_{ i :03d}' for i in range(latent_features.shape[1])]
    )


    if not len(latent_features) == len(main_continuous_resampled_date_index): raise ValueError
    else:
        print(f'len(latent_features) == len(main_continuous_resampled_date_index): True')
        print(f'len(latent_features): {len(latent_features)}')

    print(f"降维完成:")
    print(f" - 原始维度: {X_all_tensor.shape[1]}")
    print(f" - 新维度: {latent_features.shape[1]}")

    latent_features.to_csv(Specific_data_latent_features_path, index = False)

    return latent_features

In [22]:
data_to_process_dict = {
    # 'SWT_real': SWT_real,
    # 'SWT_imag': SWT_imag,
    # 'SWT_phase': SWT_phase,
    # 'SWT_magnitude': SWT_magnitude,

    # 'near_month_price': near_month_price_diffed_backward.iloc[:, :200],
    # 'far_month_price': far_month_price_diffed_backward.iloc[:, :200],
    # 'near_main_near_spread': near_main_near_spread_diffed_backward.iloc[:, :200],
    # 'near_main_far_spread': near_main_far_spread_diffed_backward.iloc[:, :200],

    'close_diffed': main_continuous_resampled_timeseries_close_diffed_backward.iloc[:, :600],
    'close_variance': main_continuous_resampled_timeseries_close_variance,
    # 'close_sampen': main_continuous_resampled_timeseries_close_sampen,
    'close_zscore': main_continuous_resampled_timeseries_close_zscore,

}

for data_set_name, data_set in data_to_process_dict.items():

    # data_set = SWT_real.copy()
    # data_set_name = 'SWT_real'
    train_data = data_set.iloc[ :split_index].dropna().values
    test_data = data_set.iloc[split_index: ].values

    # 驱动轴数据文件路径 resample_axis_data_path
    resample_axis_data_path = os.path.join(tick_granularity_resample_path, resample_axis)

    # 自编码器降维路径 VAE_dimension_reduction_path
    VAE_dimension_reduction_path = os.path.join(resample_axis_data_path, 'VAE_dimension_reduction')
    os.makedirs(VAE_dimension_reduction_path, exist_ok = True)

    # 1. 路径准备与数据分割
    (
        train_data,
        test_data,
        Specific_data_VAE_dimension_reduction_path,
        VAE_dimension_reduction_params_path,
        VAE_dimension_reduction_class_path,
        Specific_data_latent_features_path
    ) = prepare_vae_data_and_paths(
        data_set, data_set_name, VAE_dimension_reduction_path
    )

    # 2. 检测潜在特征是否已存在
    if not (
        os.path.exists(Specific_data_VAE_dimension_reduction_path) and
        os.path.exists(VAE_dimension_reduction_params_path) and
        os.path.exists(VAE_dimension_reduction_class_path) and
        os.path.exists(Specific_data_latent_features_path)
    ):

        # 3. 超参数配置
        param_grid, callbacks = configure_vae_hyperparameters(train_data)
        max_combinations = 90  # 可以根据计算资源调整
        n_splits = 5  # 定义数量 n_splits 折交叉验证
        gc.collect()

        # 4. 交叉验证寻优
        all_results, best_params, best_score = run_vae_cross_validation(
            train_data, n_splits, param_grid, callbacks, max_combinations
        )
        gc.collect()

        # 5. 训练并持久化模型
        vae_model, scaler = train_and_persist_vae_model(
            train_data, test_data, best_params,
            VAE_dimension_reduction_params_path,
            VAE_dimension_reduction_class_path
        )
        gc.collect()

        # 6. 提取并保存潜在特征
        latent_features = extract_and_save_latent_features(
            vae_model, scaler, data_set, data_set_name,
            Specific_data_latent_features_path,
            main_continuous_resampled_date_index
        )
        gc.collect()

    else:
        print(f"数据已存在，跳过处理: {Specific_data_VAE_dimension_reduction_path}")

[I 2026-02-27 18:10:51,801] A new study created in memory with name: vae_hyperparameter_optimization


使用 Optuna 进行贝叶斯优化
最大试验次数: 90


  0%|          | 0/90 [00:00<?, ?it/s]


试验 1/90
参数: {'hidden_dims': [512, 256, 128], 'latent_dim': 244, 'beta': 1e-06, 'dropout_rate': 0.19966462104925914, 'lr': 0.0001432249371823026, 'weight_decay': 2.0511104188433975e-06, 'batch_size': 128}


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=30` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

平均验证损失: 0.018701
[I 2026-02-27 20:14:24,931] Trial 88 finished with value: 0.018701372668147087 and parameters: {'hidden_dims': (224, 112), 'latent_dim': 26, 'beta': 1e-06, 'dropout_rate': 0.05465780145371265, 'lr': 0.0007817143717936964, 'weight_decay': 2.7326441589635328e-06, 'batch_size': 128}. Best is trial 4 with value: 0.016440961882472037.

试验 90/90
参数: {'hidden_dims': [224, 112], 'latent_dim': 27, 'beta': 1e-06, 'dropout_rate': 0.053537582642055485, 'lr': 0.0007784758551858116, 'weight_decay': 1.2572979289210336e-05, 'batch_size': 128}


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


平均验证损失: 0.017062
[I 2026-02-27 20:14:46,110] Trial 89 finished with value: 0.01706158760935068 and parameters: {'hidden_dims': (224, 112), 'latent_dim': 27, 'beta': 1e-06, 'dropout_rate': 0.053537582642055485, 'lr': 0.0007784758551858116, 'weight_decay': 1.2572979289210336e-05, 'batch_size': 128}. Best is trial 4 with value: 0.016440961882472037.

优化完成!
最佳参数: {'hidden_dims': (224, 112), 'latent_dim': 28, 'beta': 1e-06, 'dropout_rate': 0.07441802850159597, 'lr': 0.0004833180632488466, 'weight_decay': 7.5911048052827045e-06, 'batch_size': 128}
最佳验证损失: 0.016441


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=30` reached.


变分自编码器降维模型 VAE_dimension_reduction_model: VAE(
  (encoder): Sequential(
    (0): Linear(in_features=112, out_features=224, bias=True)
    (1): BatchNorm1d(224, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.07441802850159597, inplace=False)
    (4): Linear(in_features=224, out_features=112, bias=True)
    (5): BatchNorm1d(112, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.07441802850159597, inplace=False)
  )
  (fc_mu): Linear(in_features=112, out_features=28, bias=True)
  (fc_logvar): Linear(in_features=112, out_features=28, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=28, out_features=112, bias=True)
    (1): BatchNorm1d(112, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.07441802850159597, inplace=False)
    (4): Linear(in_features=112, out_features=224, bias=True)
    (5): BatchNorm1d(224, eps=1e-05, mo